# ORGANELLE DECLUMPING

-------

## OBJECTIVE:
In this notebook, the logic for separating organelles within the same organelle channel is outlined. Additionally, this notebook will allow the application of the declumping protocol to any organelle segmentations that have been already made and are in need of declumping to be applied to them or reapplied to them. This notebook contains a variety of methods used for declumping organelles based on the peaks of gradients found within the raw unmixed organelle channel. Finally, at the end of the notebook is a function that enables batch processing the declumping of pre-segmented organelles.

If using this notebook solely for batch processing, the following sections of code must be run in the following order prior to running the batch processing section:

1. Imports
2. Defining Highpass Filter Function
3. Defining Otsu Size Filter Function
4. Defining Watershed Declumping Function

## SUMMARY OF WORKFLOW STEPS:
 - Inputs
    - Get and load raw image
    - Get and load pre-segmented image
 - Pre-Processing
    - Perform a high-pass gaussian filter to find peaks of intensity
    - Adjust the high-pass filtered image to better fit the segmentation (optional)
 - Core-Processing
    - Perform a segmentation on the filtered image to act as seeds
 - Post-Processing
    - Perform a masked inverted watershed to segment individual organelles of the same type
 - Export
    - Export the newly declumped image

## IMPORTS:

RUN THIS

In [1]:
from pathlib import Path
import os, sys
from infer_subc.core.file_io import (list_image_files, 
                                     read_czi_image, 
                                     import_inferred_organelle, 
                                     export_inferred_organelle)
from infer_subc.core.img import masked_inverted_watershed
import napari
import numpy as np
from scipy import ndimage
from aicssegmentation.core.utils import size_filter
from skimage.morphology import opening
from skimage.filters import threshold_otsu
from skimage.measure import label
from skimage.segmentation import watershed

viewer = napari.Viewer()

-------
## Declumping Workflow

### INPUTS
Load the raw image and the pre-segmented organelle image. The pre-segmented image will be used as a mask to determine where the organelles are, and the raw image will be used to develop seeds for the peaks of the organelle locations.

#### User Inputs

##### Image Path
Here, the user should edit any of the values to correctly access the folder containing the __RAW__ image files. The naming of this file will then be used later to collect the corresponding segmentation files.

In [2]:
test_img_n = 0

in_data_path = Path("Y:/Cohen Lab/Maria Clara/2_Lab data/1_Multispectral data/2023/112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2/Deconvolved iN day21 images 082024/tiff")
seg_path = Path("Y:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmnt day21 mito D14")
seg_suffix = "-"
im_type = ".tiff"

img_file_list = list_image_files(in_data_path,im_type)
test_img_name = img_file_list[test_img_n]

out_data_path = Path("Y:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segm iNday21 D14 - declumped")
if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

In [3]:
img_data,meta_dict = read_czi_image(test_img_name)

channel_names = meta_dict['name']
img = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']

##### Desired Organelle to Declump
These user based inputs determine which organelle segmentation to access. 

The naming of __org__ must match that of the naming given to the file in the export. For example, in the infer_mitochondria segmentation notebook, the export of the segmentation has the suffix of __mito__ added to the end of the file, so if we choose __"mito"__ as the org below, we will select for the mitochondria segmentation. 

The __org_channel__ variable is used to choose the channel that the organelle is present in. For example, if the raw mitochondria channel is channel number 3 (with channel numbers starting from zero), the variable used to select for the raw mitochondria channel would be set to __3__. 

In [4]:
org = "mito"
org_channel = 3

#### Computer Found Inputs
These lines of code gather the segmentation file for the organelle chosen to be declumped above and selects specifically for the channel chosen for the organelle in the above raw unmixing image.

In [5]:
org_seg = import_inferred_organelle(name=org,meta_dict=meta_dict,out_data_path=seg_path,file_type=im_type)
org_raw = img_data[org_channel]

loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 


#### Visualize the inputs
To better understand the image, the raw organelle image and the segmented organelle image are added to the napari viewer using the below lines of code.

In [6]:
viewer.add_image(org_raw, name=f"Raw {org}", scale=scale)
viewer.add_labels(org_seg, name=f"Segmented {org}", scale=scale)

<Labels layer 'Segmented mito' at 0x20cbc151ab0>

### PRE-PROCESSING

#### Highpass Filter
The function of the highpass filter is to remove parts of an image that are lower in intensity than a desired peak, but allows incorporation of the gradients themselves into determining where the peaks are located. The inputs for the filter are purely based on the raw image. To get a highpass, we first develop the lowpass of the image by applying a gaussian filter to it, then subtract the result of the lowpass filter from the original image. 

##### Sigma Value (User Input)
The sigma value helps determine the level of the gradient taken into consideration by the highpass filter. This requires user input to help determine the best possible sigma value for the cells being analyzed.

In [7]:
sigma = 1

In [8]:
lowpass = ndimage.gaussian_filter(org_raw, sigma)
highpass = org_raw - lowpass
highpass[highpass < 0] = 0

##### Adjusting The Filtered Image (Optional)
For some images, the highpass filter on its own does not get the peaks quite perfectly. In these cases, there are several means of adjusting the filtered image to improve its ability to act as seeds for the separation of the organelles.

##### Multi Highpass (Optional)
Sometimes, a single highpass is not enough to properly declump all of the objects. Using less intense highpass filters multiple times can result in more precise separation of the peaks.

In [9]:
sigma = 1
iterations = 1

In [10]:
while iterations > 0:
    lowpass = ndimage.gaussian_filter(highpass, sigma)
    highpass = highpass - lowpass
    highpass[highpass < 0] = 0
    iterations -=1

##### Opening Filter (Optional)
The opening filter can help remove small bridges between peaks of the intensity. This filter does however have a tendency to remove bright spots in the image, leading to the possibility of removing the desired peaks.

In [25]:
open = True

In [26]:
if open:
    highpass = opening(highpass)

##### Visualize the Filter
The below code will add the filtered image to the napari viewer. There, you can view how the highpass looks and come back to this notebook to make adjustments as needed. Do note that this filter is not meant to look like a perfect separation of the organelles at this time.

In [27]:
viewer.add_image(highpass, name=f"Highpass {org}", scale=scale)

<Image layer 'Highpass mito [2]' at 0x20d411a5f90>

### Defining Highpass Filter Function

The initial step in the function is to develop the inital highpass. This is similar to the first part of the PRE-PROCESSING workflow above. This step of the function is REQUIRED, and as such, the required input values of the image and the __sigma__ must be entered for the function as a whole to run.

The second step in the function is the multihighpass. This is present in the optional section of the PRE-PROCESSING workflow. Because this section is optional, the __iterations__ variable is pre-set to 1, and requires specifically being called upon and increased if the user wishes to have more than one iteration.

The third step in the function is the opening. This is present in the optional section of the PRE-PROCESSING workflow. Because this section is optional, the __open__ variabel is pre-set to False, and requires being called upon as True if the user wishes to perform an opening.

In [28]:
def _highpass_filter(in_img: np.ndarray, sigma:float, iterations:int=1, open:bool=False) -> np.ndarray:
    #####################
    # Initial Highpass
    #####################
    lowpass = ndimage.gaussian_filter(in_img, sigma)
    highpass = in_img - lowpass
    highpass[highpass < 0] = 0

    #####################
    # Optional Multipass
    #####################
    if iterations > 1:
        for _ in range(iterations-1):
            highpass=_highpass_filter(in_img=highpass, sigma=sigma, open=False, iterations=1)
    
    #####################
    # Optional Opening
    #####################
    if open:
        highpass=opening(highpass)

    #####################
    # Return Results
    #####################
    return highpass

In [29]:
viewer.add_image(_highpass_filter(org_raw, 
                                  sigma=sigma,
                                  iterations=iterations,
                                  open=open), name=f"Highpass {org}", scale=scale)

<Image layer 'Highpass mito [3]' at 0x20d41246f80>

### CORE-PROCESSING



#### Modified Otsu Segmentation 
By utilizing an otsu segmentation, we select specifically for the peaks of the image to act as seeds for later down in the pipeline. This modified Otsu method incorporates a means to slightly adjust the results of the otsu determined threshold with an adjustment scalar value. This adjustment scalar value requires user input to be changed from anything other than 1 (default otsu thresholding) for it to make any impact on the otsu threshold.

In [30]:
thresh_adj = 0.1

In [31]:
threshold = threshold_otsu(highpass)
ots = (highpass >= (threshold*thresh_adj))

#### Size Filter
Sometimes, when making the threshold, too many objects are picked up. This results in over-declumping of organelles. By applying a size filter, objects that are too small and are potentially just noise are removed from the threshold.

##### Size (User Input)
The __min_size__ variable is used to determine the minimum size of the objects in the image. This variable requires user input of numbers 0 or greater. Using the number 0, all objects will be considered normal, and the size filter will not remove any objects.

In [32]:
min_size = 10

In [33]:
ots = size_filter(img=ots, min_size=min_size, method='3D')

#### Visualize the Threshold
The below code will add the thresholded image to the napari viewer. There, you can view how the thresholding looks and come back to this notebook to make adjustments as needed. Do note that this is not meant to look like a perfect separation of the organelles at this time.

In [34]:
viewer.add_image(ots, name=f"Thresholded Highpass {org}", scale=scale)

<Image layer 'Thresholded Highpass mito [1]' at 0x20c962f52a0>

### Defining the Otsu Size Filter

The initial step of this function is to perform the otsu thresholding on the input image. As it is optional to adjust the otsu-determined threshold, the __thresh_adj__ variable is set to default at 1, and requires the user to call upon it to change it to their desired adjustment scalar.

The second step of this function is to remove objects that are too small in size. As it is optional to do so, the __min_size__ variable is set to default at 0, and requires the user to call upon it to change it to their desired minimum size.

In [35]:
def _otsu_size_filter(in_img: np.ndarray, thresh_adj:float=1, min_size:int=0) -> np.ndarray:
    ####################
    # Otsu Thresholding
    ####################
    threshold = threshold_otsu(in_img)
    ots = (in_img >= (threshold*thresh_adj))

    ####################
    # Size Filtering
    ####################
    out_img = size_filter(img=ots, min_size=min_size, method='3D')

    ####################
    # Return Results
    ####################
    return out_img

### POST-PROCESSING

#### Watershed Declumping
Using the thresholded highpass image, we now have a grouping of "seeds" to act as markers for the watershed function. This will allow the "seeds" to spread out using the original intensity image as a sort of topography map for how easy it will be for the seeds to spread.

In [36]:
declumped = label((org_seg) + masked_inverted_watershed(org_raw, 
                                                        label(ots), 
                                                        org_seg,
                                                        method='3D'))

#### Visualize the Declumping
The below code will add the declumped image to the napari viewer. There, you can view how the declumping looks and come back to this notebook to make adjustments as needed. This is the final result of the declumping, so if you are not happy with it, take a look at the previous images and determine how you could better the seeds to improve the declumping.

In [37]:
viewer.add_labels(declumped, name=f"Declumped {org}", scale=scale)

<Labels layer 'Declumped mito [1]' at 0x20cbbeefc70>

### Defining Watershed Declumping Function

The watershed declumping function proceeds through the workflow of PRE-PROCESSING, CORE-PROCESSING, and POST-PROCESSING, allowing it to serve as the only function needed to be run by the user.

This function takes in all the variables used by the ___highpass_filter__ function and the ___otsu_size_filter__ function, in addition to one additional variable--the original organelle segmentation.

In [38]:
def _watershed_declumping(raw_img:np.ndarray, seg_img:np.ndarray, sigma:float, iterations:int=1,
                          open:bool=False, thresh_adj:float=1, min_size:int=0) -> np.ndarray:
    # PRE-PROCESSING #
    highpass = _highpass_filter(in_img=raw_img, sigma=sigma, open=open, iterations=iterations)

    # CORE-PROCESSING #
    ots = _otsu_size_filter(in_img=highpass, thresh_adj=thresh_adj, min_size=min_size)
    
    # POST-PROCESSING #
    out_img = label((seg_img) + watershed(image=(np.max(raw_img)-raw_img), 
                                          markers=label(ots), 
                                          mask=seg_img,
                                          connectivity=np.ones((3, 3, 3), bool)))
    return out_img

### EXPORT

In [39]:
out_file_n = export_inferred_organelle(declumped.astype(np.uint16), org, meta_dict, out_data_path)

saved file: 02082024_MSi08L_iN_Day21_BR3a_N02_Unmixing_0_cmle.ome-mito


-------
## Batch Processing Declumping
This section of the notebook enables batch processing of the declumping to enable declumping to be performed on multiple cells all at once. For this code to work, the __Highpass Filter__ function, __Otsu Size Filter__ function, and __Watershed Declumping__ function from the __Defining Functions__ section of this notebook all must already be run. After these three functions are run, the code in this section will work properly. 

### User Inputs

#### Existing File Paths
To make declumping batch processable, we must create a list of all the images rather than just taking in one image at a time. To do this, we need two separate file paths: a raw file path, and a segmentation file path. These are the locations of the folders that contain all the unmixed images, and the location of the folder that contains the segmented organelle images. The computer must also be told the type of file that is being searched for--usually, this file type is ".tiff".

In [40]:
raw_path=Path("Y:/Cohen Lab/Maria Clara/2_Lab data/1_Multispectral data/2023/112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2/Deconvolved iN day21 images 082024/tiff")
seg_path=Path("Y:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmnt day21 mito D14")
file_type=".tiff"

#### Output File Path
You will also need a path for the folder where the newly declumped images will be exported to.

In [41]:
out_path=Path("Y:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segm iNday21 D14 - declumped")

#### Function Inputs
Next, you will also need to determine the inputs for your batch processing. These inputs are the same inputs that were used when testing the functionality of the declumping function above. 

In [42]:
org = "mito"
org_channel = 3
sigma = 1
iterations = 1
open = True
thresh_adj = 0.1 
min_size = 10
stack = False

### Batch Process Function
Here, the function to perform the batch processing is defined. This function searches through each file in the unmixing folder, and searches through the segmentation folder for the corresponding file, and runs the declumping protocol on those files. The resulting declumped organelle image is output into the folder __"output file path"__ is defined as.

In [43]:
def declumping_batch_process(raw_path: str,
                             seg_path: str,
                             out_path: str,
                             file_type: str,
                             org: str,
                             org_channel: int,
                             sigma: float,
                             iterations: int=1,
                             open: bool=True,
                             thresh_adj: float=1,
                             min_size: int=0,
                             stack: bool=False):
    raw_path = Path(raw_path)
    seg_path = Path(seg_path)
    out_path = Path(out_path)
    
    declump_list = []
    img_file_list = list_image_files(raw_path, file_type)
    for img_f in img_file_list:
        img_data, meta_dict = read_czi_image(img_f)
        org_img = img_data[org_channel]
        org_seg = import_inferred_organelle(name=org,meta_dict=meta_dict,
                                            out_data_path=seg_path,file_type=file_type)
        declumped = _watershed_declumping(raw_img=org_img, seg_img=org_seg, sigma=sigma, iterations=iterations, 
                                          open=open, thresh_adj=thresh_adj, min_size=min_size)
        if stack:
            out_file_n = export_inferred_organelle(declumped.astype(np.uint16), 
                                                   org, meta_dict, out_path)
            viewer.add_labels(declumped)
        else: 
            out_file_n = export_inferred_organelle(declumped.astype(np.uint16), 
                                                   org, meta_dict, out_path)
        declump_list.append(out_file_n)
    return declump_list

### Running Batch Processing
This code enables the above batch process function to run. It requires no user inputs as those inputs are called upon in the below chunk of code.

In [44]:
declumped_list = declumping_batch_process(raw_path=raw_path,
                                          seg_path=seg_path,
                                          out_path=out_path,
                                          file_type=file_type,
                                          org=org,
                                          org_channel=org_channel,
                                          sigma=sigma,
                                          iterations=iterations,
                                          open=open,
                                          thresh_adj=thresh_adj,
                                          min_size=min_size,
                                          stack=stack)

loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 
saved file: 02082024_MSi08L_iN_Day21_BR3a_N02_Unmixing_0_cmle.ome-mito
loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 
saved file: 02082024_MSi08L_iN_Day21_BR3a_N03_Unmixing_0_cmle.ome-mito
loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 
saved file: 02082024_MSi08L_iN_Day21_BR3a_N04_Unmixing_0_cmle.ome-mito
loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 
saved file: 02082024_MSi08L_iN_Day21_BR3a_N05_Unmixing_0_cmle.ome-mito
loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 
saved file: 02082024_MSi08L_iN_Day21_BR3a_N06_Unmixing_0_cmle.ome-mito
loaded  inferred 3D `mito`  from Y:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmnt day21 mito D14 
saved file: 02082024_MSi08L_iN_Day21_BR

Unstack

In [ ]:
def unstack_declumped_images(out_path: str,
                             file_list: str=None,
                             declumped_path: str=None,
                             file_type: str=".tiff"):
    out_path = Path(out_path)
    if file_list != None:
        for img_f in file_list:
            img_data, meta_dict = read_czi_image(img_f)
            out_file_n = export_inferred_organelle(img_data[0], org, meta_dict, out_path)
    elif declumped_path != None:
        declumped_path = Path(declumped_path)
        img_file_list = list_image_files(declumped_path, file_type)
        for img_f in img_file_list:
            img_data, meta_dict = read_czi_image(img_f)
            out_file_n = export_inferred_organelle(img_data[0], org, meta_dict, out_path)
    else:
        raise ValueError("missing either a list of files or a path to find the files")

In [ ]:
unstack_declumped_images(out_path=out_path,
                         file_list=declumped_list)

In [ ]:
# img_file_list = list_image_files(raw_path, file_type)
# for img_f in img_file_list:
#     img_data, meta_dict = read_czi_image(img_f)
#     org_seg = import_inferred_organelle(name=org,meta_dict=meta_dict,
#                                             out_data_path=out_path,file_type=file_type)
#     viewer.add_image(img_data, scale=meta_dict["scale"])
#     viewer.add_labels(org_seg, scale=meta_dict["scale"])
#     orig_seg = import_inferred_organelle(name=org,meta_dict=meta_dict,
#                                         out_data_path=seg_path,file_type=file_type)
#     viewer.add_labels(orig_seg, scale=meta_dict["scale"])